In [ ]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import numpy as np # Necesario para una limpieza robusta de NaN

In [ ]:
DB_HOST = '127.0.0.1'
DB_USER = 'root'
DB_PASSWORD = 'AlumnaAdalab'
DB_NAME = 'music_stream_team1'
CSV1_ARTISTA = 'CSVs/artistas_lastfm.csv.csv'  
CSV2_TRACKS = 'CSVs/tracks_spotify.csv'   

In [ ]:
def crear_conexion():
    
    try:
        conn = mysql.connector.connect(
            host=DB_HOST,
            database=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD
        )
        if conn.is_connected():
            print("Conexión exitosa a MySQL.")
            return conn
    except Error as e:
        print(f"Error al conectar a MySQL: {e}")
        return None


conexion = crear_conexion()

if conexion is None:
    print(" No se pudo continuar la ejecución sin una conexión a la base de datos.")
else:
   
    cursor = conexion.cursor()

    # --- INICIO DEL PROCESO DE CARGA ---

    try:
        # ====================================================================
        # PASO 1: CARGA Y MAPEO DE GENERO_MUSICAL (Tabla sin FK)
        # ====================================================================
        print("\n--- PASO 1: Cargando Generos ---")
        
        # Carga el CSV que contiene la columna Género (CSV2)
        df_tracks_raw = pd.read_csv(CSV2_TRACKS)

        # 1. Obtener lista única de géneros (Limpieza básica de espacios)
        df_generos_unicos = df_tracks_raw[['Género']].drop_duplicates().dropna()
        df_generos_unicos['Género'] = df_generos_unicos['Género'].str.strip()
        df_generos_unicos = df_generos_unicos.rename(columns={'Género': 'nombre_genero'})
        
        # Preparar datos para la inserción
        sql_insert_genero = "INSERT INTO genero_musical (nombre_genero) VALUES (%s)"
        datos_genero = [(g,) for g in df_generos_unicos['nombre_genero']]
        
        # Insertar y crear mapeo
        cursor.executemany(sql_insert_genero, datos_genero)
        conexion.commit()
        print(f"{len(datos_genero)} géneros únicos insertados.")
        
        sql_select_generos = "SELECT id_genero, nombre_genero FROM genero_musical"
        cursor.execute(sql_select_generos)
        resultados = cursor.fetchall()
        mapa_genero = {nombre: id_ for id_, nombre in resultados}
        print("Mapeo de Géneros (Nombre -> ID) Creado.")


        # ====================================================================
        # PASO 2: CARGA Y MAPEO DE ARTISTA (Tabla sin FK)
        # ====================================================================
        print("\n--- PASO 2: Cargando Artistas ---")
        
        # 1. Preparación del DataFrame de Artistas (CSV1)
        df_artista = pd.read_csv(CSV1_ARTISTA)
        
        # Renombrar columnas
        df_artista = df_artista.rename(columns={
            'artista': 'nombre', 
            'bio_resumen': 'biografia', 
            'pais': 'pais'
            # 'oyentes' y 'reproducciones' ya tienen nombres similares
        }).drop_duplicates(subset=['nombre']) # Asumimos que el nombre es la clave

        # Columnas a insertar en MySQL
        columnas_sql_artista = ['nombre', 'biografia', 'pais', 'oyentes', 'reproducciones']
        df_artista_insert = df_artista[columnas_sql_artista]
        
        # 🔑 SOLUCIÓN: Reemplazar NaN de Pandas con None de Python
        # Esto es CRUCIAL para evitar el error 'Unknown column 'nan''
        df_artista_insert = df_artista_insert.where(pd.notnull(df_artista_insert), None) 

        # Asegurar que los valores numéricos (oyentes/reproducciones) que no son None sean del tipo correcto
        # Esto maneja el caso de que Pandas convierta la columna entera a float debido a los NaNs
        for col in ['oyentes', 'reproducciones']:
             # Convertir solo los valores que no son None a int (asume que 0 si es None es aceptable)
            df_artista_insert[col] = df_artista_insert[col].apply(
                lambda x: int(x) if pd.notnull(x) and x is not None else None
            )


        # 2. Inserción en 'artista'
        sql_insert_artista = "INSERT INTO artista (nombre, biografia, pais, oyentes, reproducciones) VALUES (%s, %s, %s, %s, %s)"
        datos_artista = [tuple(row) for row in df_artista_insert.values]
        
        cursor.executemany(sql_insert_artista, datos_artista)
        conexion.commit()
        print(f"{len(datos_artista)} artistas insertados.")
        
        # 3. Crear Mapeo de Artistas
        sql_select_artistas = "SELECT id_artista, nombre FROM artista"
        cursor.execute(sql_select_artistas)
        resultados = cursor.fetchall()
        mapa_artista = {nombre: id_ for id_, nombre in resultados}
        print("Mapeo de Artistas (Nombre -> ID) Creado.")


        # ====================================================================
        # PASO 3: PREPARACIÓN Y CARGA DE TRACKS (Tabla con FK)
        # ====================================================================
        print("\n--- PASO 3: Cargando Tracks ---")

        # El DataFrame df_tracks_raw ya está cargado del Paso 1

        # 1. Renombrar columnas (usando tu diccionario)
        df_tracks_preparado = df_tracks_raw.rename(columns={
            'ID': 'id_track',
            'Nombre': 'nombre',
            'Tipo': 'tipo',
            'Año': 'anio_lanzamiento',
            'Género': 'nombre_genero_fk',
            'Artista': 'nombre_artista_fk' })

        # 2. Aplicar mapeos 
        df_tracks_preparado['id_genero'] = df_tracks_preparado['nombre_genero_fk'].astype(str).str.strip().map(mapa_genero)
        df_tracks_preparado['id_artista'] = df_tracks_preparado['nombre_artista_fk'].astype(str).str.strip().map(mapa_artista)

        # 3. Selección final de columnas 
        columnas_tracks_sql = ['id_track', 'nombre', 'tipo', 'anio_lanzamiento', 'id_genero', 'id_artista']
        df_tracks_insert = df_tracks_preparado[columnas_tracks_sql].copy() # <--- ¡IMPORTANTE: Usar .copy() para evitar advertencia!

        # 4. Limpieza final y conversión de tipos
        # Eliminar filas donde el FK (id_genero/id_artista) no se encontró (es NaN)
        df_tracks_insert.dropna(subset=['id_genero', 'id_artista'], inplace=True)

      
        df_tracks_insert = df_tracks_insert.replace({np.nan: None}) 

        # Convertir IDs a int (están en float por el mapeo)
        df_tracks_insert['id_genero'] = df_tracks_insert['id_genero'].astype(int)
        df_tracks_insert['id_artista'] = df_tracks_insert['id_artista'].astype(int)

        # 5. Inserción en 'tracks'
        sql_insert_track = "INSERT IGNORE INTO tracks (id_track, nombre, tipo, anio_lanzamiento, id_genero, id_artista) VALUES (%s, %s, %s, %s, %s, %s)"
        datos_tracks = [tuple(row) for row in df_tracks_insert.values] # Convierte los valores (que ahora tienen None) a tuplas

        cursor.executemany(sql_insert_track, datos_tracks)
        conexion.commit()
        print(f"{len(datos_tracks)} tracks insertados (excluyendo aquellos sin ID de Artista/Género).")

    except Error as e:
        print(f"Ocurrió un error grave en la inserción: {e}")
        conexion.rollback() # Revierte todas las operaciones si hay un error
    finally:
        # Cierre de recursos
        if 'cursor' in locals() and cursor:
            cursor.close()
        if conexion and conexion.is_connected():
            conexion.close()
            print("\n🔌 Conexión a MySQL cerrada.")